<a href="https://colab.research.google.com/github/nandhadhev/machine-learning/blob/main/exp_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier

In [4]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [6]:
# Load Titanic dataset from an online source
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)

In [7]:
# Select features
features = [
    "Pclass", "Sex", "Age", "SibSp",
    "Parch", "Fare", "Embarked"
]

In [9]:
# Select lowercase features to match the loaded dataset's columns
features_lower = [f.lower() for f in features]
X = df[features_lower]
y = df["survived"]

In [10]:
categorical_features = ["Sex", "Embarked"]

In [12]:
# Define feature groups with lowercase names matching X
numeric_features = ["pclass", "age", "sibsp", "parch", "fare"]
categorical_features_lower = ["sex", "embarked"]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features_lower)
    ]
)

In [13]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=42, stratify=y
)

In [14]:
# Base Decision Tree
base_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)


In [15]:
# Bagging
bagging = BaggingClassifier(
    estimator=base_model,
    n_estimators=50,
    random_state=42
)

In [16]:
bagging_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", bagging)
])

In [17]:
bagging_pipeline.fit(X_train, y_train)
bag_pred = bagging_pipeline.predict(X_test)


In [18]:
# AdaBoost
boosting = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=2,
        random_state=42
    ),
    n_estimators=50,
    random_state=42
)

In [19]:
boosting_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", boosting)
])


In [21]:
from sklearn.impute import SimpleImputer

# Redefine preprocessor with imputer steps to handle missing values
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("pass", "passthrough")
])

cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, numeric_features),
        ("cat", cat_transformer, categorical_features_lower)
    ]
)

# Re-create and fit the boosting pipeline
boosting_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", boosting)
])

boosting_pipeline.fit(X_train, y_train)
boost_pred = boosting_pipeline.predict(X_test)

In [22]:
# Evaluation function
def evaluate_model(name, y_true, y_pred):
    print("\n", name)

    print("Accuracy:",
          round(accuracy_score(y_true, y_pred), 4))

    print("Precision:",
          round(precision_score(y_true, y_pred), 4))

    print("Recall:",
          round(recall_score(y_true, y_pred), 4))

    print("F1-score:",
          round(f1_score(y_true, y_pred), 4))

# Evaluate
evaluate_model("Bagging", y_test, bag_pred)
evaluate_model("Boosting (AdaBoost)", y_test, boost_pred)


 Bagging
Accuracy: 0.7989
Precision: 0.8667
Recall: 0.5652
F1-score: 0.6842

 Boosting (AdaBoost)
Accuracy: 0.7765
Precision: 0.7636
Recall: 0.6087
F1-score: 0.6774
